In [0]:
employee=spark.read.csv("/Volumes/workspace/employee_project/employee_files/employee_data_quality_test2.csv",header=True,inferSchema=True)

In [0]:
employee.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- designation: string (nullable = true)
 |-- manager_id: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- joining_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- employee_status: string (nullable = true)



### Invalid salary

In [0]:
from pyspark.sql.functions import col,current_date,to_date
invalid_salary=employee.filter(col("salary")<0)
invalid_salary.show(truncate=False)

+-----------+----------+---------+-------------+-----------+----------+------+------------+----------+---------------+
|employee_id|first_name|last_name|department   |designation|manager_id|salary|joining_date|city      |employee_status|
+-----------+----------+---------+-------------+-----------+----------+------+------------+----------+---------------+
|EMP100005  |Nachiket  |Bath     |HR           |Manager    |EMP100001 |-55000|NULL        |Hyderabad |Inactive       |
|EMP100012  |Owen      |Sankar   |Sales        |Manager    |EMP100001 |-72000|NULL        |Delhi     |Inactive       |
|EMP100020  |Avi       |Chander  |IT           |Analyst    |EMP100001 |-48000|NULL        |Pune      |Active         |
|EMP100025  |Sneha     |Munshi   |IT           |Engineer   |EMP100001 |-65000|NULL        |Chennai   |Inactive       |
|EMP200001  |Rahul     |Kumar    |IT           |Developer  |EMP100001 |-55000|NULL        |Chennai   |Active         |
|EMP200006  |Meena     |Iyer     |Alien Finance|

### Invalid department

In [0]:
valid_department=['Sales','HR','IT','Marketing','Finance']
invalid_department=employee.filter(~col("department").isin(valid_department))
invalid_department.show()

+-----------+----------+---------+-------------+------------+----------+------+------------+---------+---------------+
|employee_id|first_name|last_name|   department| designation|manager_id|salary|joining_date|     city|employee_status|
+-----------+----------+---------+-------------+------------+----------+------+------------+---------+---------------+
|  EMP200002|     Priya|   Sharma|     Alien HR|HR Executive| EMP100002| 62000|        NULL|Bengaluru|         Active|
|  EMP200006|     Meena|     Iyer|Alien Finance|     Manager| EMP888888|-72000|        NULL|    Delhi|    On Vacation|
+-----------+----------+---------+-------------+------------+----------+------+------------+---------+---------------+



### Invalid date

In [0]:
invalid_date=employee.filter(to_date(col('joining_date'),'yyyy-mm-dd')>current_date())
invalid_date.show()

+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+
|employee_id|first_name|last_name|department|designation|manager_id|salary|joining_date|     city|employee_status|
+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+
|  EMP100005|  Nachiket|     Bath|        HR|    Manager| EMP100001|-55000|  2040-02-14|Hyderabad|       Inactive|
|  EMP100007|Harinakshi|     Kota|        HR|    Manager| EMP100001|408618|  2045-02-21|     Pune|       Inactive|
|  EMP200005|      Arun|    Patel|        IT|   Engineer| EMP100005| 71000|  2045-10-25|   Mumbai|         Active|
+-----------+----------+---------+----------+-----------+----------+------+------------+---------+---------------+



### Invalid status

In [0]:
valid_status=['Active','Inactive']
invalid_status=employee.filter(~col("employee_status").isin(valid_status))
invalid_status.show()

+-----------+----------+---------+-------------+-----------+----------+------+------------+-----+---------------+
|employee_id|first_name|last_name|   department|designation|manager_id|salary|joining_date| city|employee_status|
+-----------+----------+---------+-------------+-----------+----------+------+------------+-----+---------------+
|  EMP200004|     Sneha|      Rao|        Sales|  Executive| EMP100004| 58000|  2025-04-05| Pune|        Unknown|
|  EMP200006|     Meena|     Iyer|Alien Finance|    Manager| EMP888888|-72000|  2025-06-18|Delhi|    On Vacation|
|  EMP200007|   Karthik|     Nair|    Marketing| Specialist| EMP100007| 64000|  2025-07-25|Kochi|     ACTIVE_NOW|
+-----------+----------+---------+-------------+-----------+----------+------+------------+-----+---------------+



### Invalid manager id

In [0]:
invalid_manager=employee.filter(col("manager_id")!='EMP100001')
invalid_manager.show()

+-----------+----------+---------+-------------+------------+----------+------+------------+----------+---------------+
|employee_id|first_name|last_name|   department| designation|manager_id|salary|joining_date|      city|employee_status|
+-----------+----------+---------+-------------+------------+----------+------+------------+----------+---------------+
|  EMP200002|     Priya|   Sharma|     Alien HR|HR Executive| EMP100002| 62000|  2025-02-20| Bengaluru|         Active|
|  EMP200003|    Vikram|    Singh|      Finance|     Analyst| EMP999999| 68000|  2025-03-10| Hyderabad|         Active|
|  EMP200004|     Sneha|      Rao|        Sales|   Executive| EMP100004| 58000|  2025-04-05|      Pune|        Unknown|
|  EMP200005|      Arun|    Patel|           IT|    Engineer| EMP100005| 71000|  2045-10-25|    Mumbai|         Active|
|  EMP200006|     Meena|     Iyer|Alien Finance|     Manager| EMP888888|-72000|  2025-06-18|     Delhi|    On Vacation|
|  EMP200007|   Karthik|     Nair|    Ma

In [0]:
final_table=employee.withColumn("Valid Salary",col("salary")>=0).withColumn("Valid Status",col("employee_status").isin(valid_status)).withColumn("Valid Department",col("department").isin(valid_department)).withColumn("Valid Date",to_date(col('joining_date'),'yyyy-mm-dd')<=current_date()).withColumn("Valid Manager",col("manager_id")=='EMP100001')
final_table.show()

+-----------+----------+---------+----------+-----------+----------+-------+------------+----------+---------------+------------+------------+----------------+----------+-------------+
|employee_id|first_name|last_name|department|designation|manager_id| salary|joining_date|      city|employee_status|Valid Salary|Valid Status|Valid Department|Valid Date|Valid Manager|
+-----------+----------+---------+----------+-----------+----------+-------+------------+----------+---------------+------------+------------+----------------+----------+-------------+
|  EMP100001|   Devansh|  Thakkar|     Sales|    Manager|      NULL| 578082|  2025-11-26|   Chennai|         Active|        true|        true|            true|      true|         NULL|
|  EMP100002|   Yashoda|    Batra|     Sales|   Engineer| EMP100001| 372132|  2024-01-29|   Chennai|       Inactive|        true|        true|            true|      true|         true|
|  EMP100003|     Meera| Banerjee|     Sales|   Engineer| EMP100001| 598830

In [0]:
valid_columns=final_table.withColumn("is_valid",col("Valid Salary")&col("Valid Date")&col("Valid Department")&col("Valid Status")&col("Valid Manager"))
valid_columns.show()

+-----------+----------+---------+----------+-----------+----------+-------+------------+----------+---------------+------------+------------+----------------+----------+-------------+--------+
|employee_id|first_name|last_name|department|designation|manager_id| salary|joining_date|      city|employee_status|Valid Salary|Valid Status|Valid Department|Valid Date|Valid Manager|is_valid|
+-----------+----------+---------+----------+-----------+----------+-------+------------+----------+---------------+------------+------------+----------------+----------+-------------+--------+
|  EMP100001|   Devansh|  Thakkar|     Sales|    Manager|      NULL| 578082|  2025-11-26|   Chennai|         Active|        true|        true|            true|      true|         NULL|    NULL|
|  EMP100002|   Yashoda|    Batra|     Sales|   Engineer| EMP100001| 372132|  2024-01-29|   Chennai|       Inactive|        true|        true|            true|      true|         true|    true|
|  EMP100003|     Meera| Baner